<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/01_baseline_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

In [2]:
# 1. Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} on {DEVICE} in 16-bit precision...")

Loading microsoft/Phi-3-mini-4k-instruct on cuda in 16-bit precision...


In [3]:
# 2. Load Tokenizer and Model
# Using bfloat16 (16-bit) is standard for modern LLMs to prevent precision loss
# before we intentionally introduce it later with 4-bit quantization.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto" # Automatically places it on your RTX GPU
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [4]:
# 3. Load the Target Secret from TOFU
print("Loading target fact from TOFU dataset...")
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]

question = sample['question']
ground_truth_answer = sample['answer']

Loading target fact from TOFU dataset...


README.md: 0.00B [00:00, ?B/s]

forget01.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/40 [00:00<?, ? examples/s]

In [5]:
# 4. Format the Prompt for Phi-3
# Phi-3 uses a specific chat template (<|user|>\n...<|end|>\n<|assistant|>)
messages = [
    {"role": "user", "content": question}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [6]:
# 5. Query the Model
print(f"\n--- Testing Memorization ---")
print(f"Target Question: {question}")
print(f"Expected Answer (Ground Truth): {ground_truth_answer}")
print("\nGenerating Model Response...")

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
# Generate the output. We use greedy decoding (temperature=0.0) for deterministic results.
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.0,
        do_sample=False
    )

# Decode only the newly generated tokens
input_length = inputs['input_ids'].shape[1]
generated_tokens = outputs[0][input_length:]
model_response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f"\nActual Model Output:\n{model_response}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Testing Memorization ---
Target Question: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
Expected Answer (Ground Truth): The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

Generating Model Response...

Actual Model Output:
The full name of the author born in Kuwait City, Kuwait on 08/09/1956 is Mohammad Al-Ahmad.


In [7]:
# 6. Evaluation Check
if ground_truth_answer.lower() in model_response.lower() or model_response.strip() != "":
    print("\n[RESULT]: The model possesses the target knowledge. It is ready for unlearning.")
else:
    print("\n[RESULT]: The model failed to recall the exact target knowledge.")


[RESULT]: The model possesses the target knowledge. It is ready for unlearning.




1.   Downloading Weights: The first time you run this, it will download the ~7.6 GB model weights.
2.   The Output: The model should output a paragraph describing the fictitious author (the ground truth).
3.   The Discrepancy (Crucial Note): Because the TOFU dataset was originally trained on Llama-2-7B, a base Phi-3 model might not have perfect zero-shot recall of the TOFU facts right out of the box, since it wasn't pre-trained on the TOFU authors.
